The task request is to "Task is to simulate as close to a baseload profile as possible, with no more than 10% of production above baseload to be curtailed."

We will use the information in the wind and solar production data to fine the optimal scalign up and down of each resource order to have a combined production profile that is as close as possible to a stable baseload, while ensuring that no more than 10% of the total energy produced is curtailed (i.e. above the baseload).


With the function  B = find_baseload(P, target_curtailment=0.10) we are already optimizign for the curtailment ratio, so the main grid search will be focused on finding the best capacities for solar and wind (S and W) that maximize the baseload.

An issue that i encurred is that some combinations of S and W that yes optimize to have a high baseload, respect the curtailment constraint, but have a very low daily average production. This is not a desirable solution in the real world, as we want to ensure that we are actually producing a significant amount of energy, CLOSE to the baseload, not just optimizing for a high baseload with very low production.

To address this "real word issue" of followign the baseload with a decent production, I added an additional constraint that the daily average production must be at least 70% of the baseload. This way we ensure that we are not just finding a high baseload with very low production, but rather a good balance between the two.

Limitations:

Ideally we would like the power poduction not only to be on average near the baseload but nead the baseload the whole time. In this situation unfortunatelly putting a constrain in that area leads to an impossible solution.
Why? There are some instances where production is zero o very low for both solar and wind, windless nights.

In this case no matter how we scale up or down our resouces we will never get close to the promised baseload. 

Ideally we would need:
1) A storage system, batteries for storing energy when none is produced 
2) Alternative power sources, coal, nuclear, gas...
3) Ignore the "outliers" and optimize the algoritm without them, which i decided to avoid doing it since i think we cannot ignore them.

In [17]:
import numpy as np
import pandas as pd

pd.set_option("display.max_columns", 200)
pd.set_option("display.width", 200)

# Inputs (override if needed)
WIND_XLSX_PATH = "Wind.xlsx"
SOLAR_CSV_PATH = "Solar.CSV"

# Outputs (same names as scripts)
WIND_RAW_OUT = "Wind_raw.csv"
WIND_AVG_OUT = "Wind.csv"
OVERALL_OUT = "Overall.csv"
GRID_LOG_OUT = "grid_search_log.csv"
OVERALL_BASELOAD_OUT = "Overall_with_baseload.csv"

### 1) Load wind Excel (`Wind.xlsx`)

In [18]:
df_wind = pd.read_excel(WIND_XLSX_PATH, header=2)
df_wind = df_wind.drop(columns=["Unnamed: 25"], errors="ignore")

# First column is the date
first_col = df_wind.columns[0]
df_wind = df_wind.rename(columns={first_col: "Date"})
df_wind["Date"] = pd.to_datetime(df_wind["Date"])

df_wind.head()

,Date,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24
0,2020-01-01,28.545,26.136,21.021,31.449,21.879,35.508,41.778,38.082,41.910,41.646,41.877,41.910,41.283,40.425,38.907,38.544,35.343,40.887,37.917,20.361,22.242,25.542,16.731,20.361
1,2020-01-02,27.621,34.353,33.660,39.633,37.686,40.887,41.910,41.910,41.943,41.910,41.910,41.943,41.910,41.910,41.910,41.943,41.217,34.881,41.778,41.910,41.910,41.943,39.699,40.029
2,2020-01-03,40.656,40.755,41.019,41.283,41.283,40.722,34.617,29.370,32.505,37.983,39.732,41.811,41.778,41.910,41.943,41.910,41.910,41.943,41.910,41.910,41.910,41.910,38.775,38.709
3,2020-01-04,38.511,39.270,28.677,35.772,39.798,40.458,41.943,41.679,41.910,41.943,41.910,41.910,41.943,41.910,41.910,29.766,39.699,41.943,41.910,41.910,41.943,41.910,39.765,40.095
4,2020-01-05,39.138,38.478,41.151,40.623,40.755,40.656,41.943,39.930,34.452,37.653,21.384,17.292,14.289,6.501,2.706,10.461,10.362,16.731,17.886,24.783,29.997,25.113,19.305,26.565


### 2) Wide → long (one row per hour) + build hourly timestamps

In [3]:
df_melted = df_wind.melt(id_vars="Date", var_name="Hour", value_name="Production, KWh")

# Use actual hour value from the column (1..24 -> 0..23)
df_melted["Hour"] = pd.to_numeric(df_melted["Hour"], errors="coerce")
df_melted["Date"] = df_melted["Date"] + pd.to_timedelta(df_melted["Hour"] - 1, unit="h")

df_melted[["Date", "Production, KWh"]].sort_values(by="Date").head()

,Date,"Production, KWh"
0,2020-01-01 00:00:00,28.545
1096,2020-01-01 01:00:00,26.136
2192,2020-01-01 02:00:00,21.021
3288,2020-01-01 03:00:00,31.449
4384,2020-01-01 04:00:00,21.879


### 3) Export wind raw (`Wind_raw.csv`)

In [4]:
df_raw = df_melted[["Date", "Production, KWh"]].dropna().sort_values("Date")
df_raw_out = df_raw.copy()
df_raw_out["Date"] = df_raw_out["Date"].dt.strftime("%d/%m/%Y %H:%M")
df_raw_out.to_csv(WIND_RAW_OUT, index=False)

print(f"Wrote {WIND_RAW_OUT} ({len(df_raw_out)} rows)")
df_raw_out.head()

Wrote Wind_raw.csv (26302 rows)


,Date,"Production, KWh"
0,01/01/2020 00:00,28.545
1096,01/01/2020 01:00,26.136
2192,01/01/2020 02:00,21.021
3288,01/01/2020 03:00,31.449
4384,01/01/2020 04:00,21.879


### 4) Average wind profile across years (year-agnostic) → `Wind.csv`

In [5]:
df_melted_nn = df_melted.dropna(subset=["Production, KWh"]).copy()

df_melted_nn["Month"] = df_melted_nn["Date"].dt.month
df_melted_nn["Day"] = df_melted_nn["Date"].dt.day
df_melted_nn["HourOfDay"] = df_melted_nn["Date"].dt.hour

grouped = (
    df_melted_nn.groupby(["Month", "Day", "HourOfDay"], as_index=False)["Production, KWh"].mean()
)

synthetic_dt = pd.to_datetime(
    dict(
        year=2000,
        month=grouped["Month"],
        day=grouped["Day"],
        hour=grouped["HourOfDay"],
    )
)
grouped["Date"] = synthetic_dt.dt.strftime("%d/%m %H:%M")

wind_avg = grouped.sort_values(["Month", "Day", "HourOfDay"])[["Date", "Production, KWh"]]
wind_avg.to_csv(WIND_AVG_OUT, index=False)

print(f"Wrote {WIND_AVG_OUT} ({len(wind_avg)} rows)")
wind_avg.head()

Wrote Wind.csv (8784 rows)


,Date,"Production, KWh"
0,01/01 00:00,16.676
1,01/01 01:00,16.423
2,01/01 02:00,15.202
3,01/01 03:00,22.341
4,01/01 04:00,18.788


### 5) Build `Overall.csv` from `Solar.CSV` + averaged wind

In [6]:
solar = pd.read_csv(SOLAR_CSV_PATH)
solar["Date"] = pd.to_datetime(solar["Date"], dayfirst=True)
solar["Month"] = solar["Date"].dt.month
solar["Day"] = solar["Date"].dt.day
solar["HourOfDay"] = solar["Date"].dt.hour

merged = solar.merge(
    grouped[["Month", "Day", "HourOfDay", "Production, KWh"]],
    on=["Month", "Day", "HourOfDay"],
    how="left",
    suffixes=("_solar", "_wind"),
)

overall = merged[["Date", "Production, KWh_solar", "Production, KWh_wind"]].rename(
    columns={
        "Production, KWh_solar": "Solar Production, KWh",
        "Production, KWh_wind": "Wind Production, KWh (avg)",
    }
)

overall.to_csv(OVERALL_OUT, index=False)
print(f"Wrote {OVERALL_OUT} ({len(overall)} rows)")
overall.head()

Wrote Overall.csv (8760 rows)


,Date,"Solar Production, KWh","Wind Production, KWh (avg)"
0,1990-01-01 00:00:00,0.0,16.676
1,1990-01-01 01:00:00,0.0,16.423
2,1990-01-01 02:00:00,0.0,15.202
3,1990-01-01 03:00:00,0.0,22.341
4,1990-01-01 04:00:00,0.0,18.788


### 6) Baseload + curtailment optimization (same logic as `prepare_baseload_curtailment.py`)

#### 6a) Define curtailment + baseload helpers

We measure curtailment as energy above a candidate baseload and compute the curtailment ratio (curtailed / total). Then we use a binary search (`find_baseload`) to find the **highest** baseload such that the curtailment ratio stays at or below the 10% target.

In [7]:
from typing import Tuple, List, Dict


def compute_curtailment_ratio(production, baseload):
    excess = np.maximum(production - baseload, 0)
    curtailed_energy = np.sum(excess)
    total_energy = np.sum(production)

    if total_energy <= 0:
        return 0.0

    return curtailed_energy / total_energy


def find_baseload(production, target_curtailment=0.10):
    if len(production) == 0 or np.sum(production) <= 0:
        return 0.0

    lo = 0.0
    hi = float(np.max(production))

    for _ in range(100):
        mid = 0.5 * (lo + hi)
        ratio = compute_curtailment_ratio(production, mid)

        if ratio > target_curtailment:
            lo = mid
        else:
            hi = mid

        if hi - lo < 1e-8:
            break

    return 0.5 * (lo + hi)

#### 6b) Load `Overall.csv` and validate inputs

We read the merged solar+wind dataset, locate the expected columns, coerce the timestamp column to datetime, and extract the solar/wind series as NumPy arrays. Basic checks (NaNs, all-zero series) prevent running the optimization on invalid data.

In [8]:
def load_and_prepare_overall(path: str = "Overall.csv"):
    print(f"Loading input data from {path} ...")
    overall_df = pd.read_csv(path)
    print(f"Loaded {len(overall_df)} rows.")

    cols = {c.lower(): c for c in overall_df.columns}
    date_col = cols.get("date", overall_df.columns[0])
    solar_col = cols.get("solar production, kwh", None)
    wind_col = cols.get("wind production, kwh (avg)", None)

    if solar_col is None or wind_col is None:
        print("ERROR: Cannot find solar and wind columns")
        print(f"Available columns: {list(overall_df.columns)}")
        return None, None, None, None, None, None

    if not pd.api.types.is_datetime64_any_dtype(overall_df[date_col]):
        overall_df[date_col] = pd.to_datetime(overall_df[date_col], errors="coerce")

    solar_raw = overall_df[solar_col].astype(float).to_numpy()
    wind_raw = overall_df[wind_col].astype(float).to_numpy()

    if np.isnan(solar_raw).any() or np.isnan(wind_raw).any():
        print("ERROR: Data contains NaN values")
        return None, None, None, None, None, None

    if np.max(solar_raw) <= 0 or np.max(wind_raw) <= 0:
        print("ERROR: No positive values in solar or wind data")
        return None, None, None, None, None, None

    return overall_df, date_col, solar_col, wind_col, solar_raw, wind_raw

#### 6c) Grid search for the best (Solar MW, Wind MW)

We try a grid of scaling factors for solar and wind, compute the maximum feasible baseload under the 10% curtailment constraint, and keep the best candidate that also meets the additional realism constraint: average production is at least 70% of the baseload. We also log every combination to `grid_search_log.csv`.

In [9]:
def search_best_capacities(
    solar_raw: np.ndarray,
    wind_raw: np.ndarray,
) -> Tuple[float, float, float]:
    print("Starting capacity grid search over S,W in [0,200] MW (step 5 MW) ...")

    S_values = np.linspace(0, 200, 41)
    W_values = np.linspace(0, 200, 41)

    best_B = -np.inf
    best_S = 0.0
    best_W = 0.0

    grid_log: List[Dict[str, float]] = []

    for S in S_values:
        for W in W_values:
            if S == 0 and W == 0:
                continue

            P = S * solar_raw + W * wind_raw
            B = find_baseload(P, target_curtailment=0.10)

            daily_avg_production = float(np.mean(P))
            daily_error_pct = (daily_avg_production - B) / B if B > 0 else 0.0

            grid_log.append(
                {
                    "S_MW": float(S),
                    "W_MW": float(W),
                    "Baseload_MW": float(B),
                    "DailyAvgProd_MW": daily_avg_production,
                    "DailyErrorPct": daily_error_pct,
                }
            )

            if B > best_B and daily_avg_production >= 0.7 * B:
                print(
                    "New best candidate -> "
                    "B={:.2f} MW, S={:.1f} MW, W={:.1f} MW, "
                    "avg_prod={:.2f} MW".format(B, S, W, daily_avg_production)
                )
                best_B = B
                best_S = S
                best_W = W

    if grid_log:
        print(f"Writing grid search log to {GRID_LOG_OUT} ...")
        df_log = pd.DataFrame(grid_log)
        df_log.to_csv(GRID_LOG_OUT, index=False)

    if best_B <= 0:
        print("ERROR: Could not find valid solution")
        return 0.0, 0.0, 0.0

    print(
        "Final choice -> "
        "B={:.2f} MW, S={:.1f} MW, W={:.1f} MW".format(best_B, best_S, best_W)
    )
    return best_B, best_S, best_W

#### 6d) Build the final output table and export

Using the best capacities found, we compute the combined production profile, derive per-hour shortfall vs baseload, compute daily averages, then write `Overall_with_baseload.csv` with the key columns ordered up front.

In [10]:
def build_output_dataframe(
    overall_df: pd.DataFrame,
    date_col: str,
    solar_col: str,
    wind_col: str,
    solar_raw: np.ndarray,
    wind_raw: np.ndarray,
    best_B: float,
    best_S: float,
    best_W: float,
) -> pd.DataFrame:
    P_optimal = best_S * solar_raw + best_W * wind_raw

    curtailment = np.maximum(P_optimal - best_B, 0.0)
    curtailment_ratio = np.zeros_like(P_optimal, dtype=float)
    mask = P_optimal > 0
    curtailment_ratio[mask] = curtailment[mask] / P_optimal[mask]

    overall_df["SolarCap_MW"] = np.round(best_S, 2)
    overall_df["WindCap_MW"] = np.round(best_W, 2)
    overall_df["Baseload_MW"] = np.round(best_B, 2)
    overall_df["SolarScaled"] = np.round(best_S * solar_raw, 2)
    overall_df["WindScaled"] = np.round(best_W * wind_raw, 2)
    overall_df["ProdCombined"] = np.round(P_optimal, 2)

    prod_error = (best_B - P_optimal) / best_B
    overall_df["HourlyShortfall"] = np.round(prod_error, 4)

    day_index = overall_df[date_col].dt.floor("D")
    overall_df["DailyAvgProd"] = overall_df.groupby(day_index)["ProdCombined"].transform("mean").round(2)
    overall_df["DailyErrorPct"] = np.round(
        (overall_df["DailyAvgProd"] - overall_df["Baseload_MW"]) / overall_df["Baseload_MW"],
        4,
    )

    overall_df[wind_col] = np.round(overall_df[wind_col].astype(float), 2)

    preferred_cols = [
        date_col,
        solar_col,
        wind_col,
        "SolarCap_MW",
        "WindCap_MW",
        "Baseload_MW",
        "SolarScaled",
        "WindScaled",
        "ProdCombined",
        "HourlyShortfall",
        "DailyAvgProd",
        "DailyErrorPct",
    ]
    ordered_cols = [c for c in preferred_cols if c in overall_df.columns] + [
        c for c in overall_df.columns if c not in preferred_cols
    ]

    print(f"Writing results to {OVERALL_BASELOAD_OUT} ...")
    overall_df.to_csv(OVERALL_BASELOAD_OUT, index=False, columns=ordered_cols)
    print("Done.")

    return overall_df

### 7) Run optimization and write `grid_search_log.csv` + `Overall_with_baseload.csv`

In [11]:
(
    overall_df,
    date_col,
    solar_col,
    wind_col,
    solar_raw,
    wind_raw,
) = load_and_prepare_overall(OVERALL_OUT)

if overall_df is None:
    raise RuntimeError("Overall data could not be loaded/prepared")

best_B, best_S, best_W = search_best_capacities(solar_raw=solar_raw, wind_raw=wind_raw)

if best_B <= 0:
    raise RuntimeError("No valid baseload solution found")

result_df = build_output_dataframe(
    overall_df=overall_df,
    date_col=date_col,
    solar_col=solar_col,
    wind_col=wind_col,
    solar_raw=solar_raw,
    wind_raw=wind_raw,
    best_B=best_B,
    best_S=best_S,
    best_W=best_W,
)

result_df.head()

Loading input data from Overall.csv ...
Loaded 8760 rows.
Starting capacity grid search over S,W in [0,200] MW (step 5 MW) ...
New best candidate -> B=114.35 MW, S=0.0 MW, W=5.0 MW, avg_prod=85.36 MW
New best candidate -> B=228.71 MW, S=0.0 MW, W=10.0 MW, avg_prod=170.71 MW
New best candidate -> B=343.06 MW, S=0.0 MW, W=15.0 MW, avg_prod=256.07 MW
New best candidate -> B=457.42 MW, S=0.0 MW, W=20.0 MW, avg_prod=341.42 MW
New best candidate -> B=571.77 MW, S=0.0 MW, W=25.0 MW, avg_prod=426.78 MW
New best candidate -> B=686.13 MW, S=0.0 MW, W=30.0 MW, avg_prod=512.13 MW
New best candidate -> B=800.48 MW, S=0.0 MW, W=35.0 MW, avg_prod=597.49 MW
New best candidate -> B=914.84 MW, S=0.0 MW, W=40.0 MW, avg_prod=682.84 MW
New best candidate -> B=1029.19 MW, S=0.0 MW, W=45.0 MW, avg_prod=768.20 MW
New best candidate -> B=1143.55 MW, S=0.0 MW, W=50.0 MW, avg_prod=853.55 MW
New best candidate -> B=1257.90 MW, S=0.0 MW, W=55.0 MW, avg_prod=938.91 MW
New best candidate -> B=1372.25 MW, S=0.0 MW, W

,Date,"Solar Production, KWh","Wind Production, KWh (avg)",SolarCap_MW,WindCap_MW,Baseload_MW,SolarScaled,WindScaled,ProdCombined,HourlyShortfall,DailyAvgProd,DailyErrorPct
0,1990-01-01 00:00:00,0.0,16.68,0.0,200.0,4574.18,0.0,3335.2,3335.2,0.2709,3466.01,-0.2423
1,1990-01-01 01:00:00,0.0,16.42,0.0,200.0,4574.18,0.0,3284.6,3284.6,0.2819,3466.01,-0.2423
2,1990-01-01 02:00:00,0.0,15.20,0.0,200.0,4574.18,0.0,3040.4,3040.4,0.3353,3466.01,-0.2423
3,1990-01-01 03:00:00,0.0,22.34,0.0,200.0,4574.18,0.0,4468.2,4468.2,0.0232,3466.01,-0.2423
4,1990-01-01 04:00:00,0.0,18.79,0.0,200.0,4574.18,0.0,3757.6,3757.6,0.1785,3466.01,-0.2423


### 8) Extend with an ideal battery (using curtailed energy)

In this section we imagine a perfect battery that stores all curtailed energy (production above baseload) and discharges it later during deficits (production below baseload). The goal is to see how much this simple storage layer can reduce windless-night shortfalls and improve effective baseload coverage.

In [12]:
import numpy as np

# We assume the previous cells have already produced result_df, best_B, best_S, best_W, and date_col
P = result_df["ProdCombined"].to_numpy(dtype=float)
B = result_df["Baseload_MW"].to_numpy(dtype=float)

n = len(P)
surplus = np.maximum(P - B, 0.0)   # potential curtailment → battery charge
raw_deficit = np.maximum(B - P, 0.0)

soc = np.zeros(n, dtype=float)
charge = np.zeros(n, dtype=float)
discharge = np.zeros(n, dtype=float)

for t in range(n):
    prev_soc = soc[t - 1] if t > 0 else 0.0
    if surplus[t] > 0:
        # charge battery with all surplus
        charge[t] = surplus[t]
        soc[t] = prev_soc + charge[t]
    elif raw_deficit[t] > 0:
        # discharge up to available energy
        discharge[t] = min(raw_deficit[t], prev_soc)
        soc[t] = prev_soc - discharge[t]
    else:
        soc[t] = prev_soc

# Effective production after storage: we clip the original profile to baseload when charging,
# and add discharge back in hours with deficits.
EffectiveProd = P + discharge - charge
ResidualDeficit = np.maximum(B - EffectiveProd, 0.0)

result_df["Battery_Charge"] = np.round(charge, 3)
result_df["Battery_Discharge"] = np.round(discharge, 3)
result_df["Battery_SoC"] = np.round(soc, 3)
result_df["EffectiveProd"] = np.round(EffectiveProd, 2)
result_df["ResidualDeficit"] = np.round(ResidualDeficit, 3)

result_df[[date_col, "ProdCombined", "Baseload_MW", "Battery_Charge", "Battery_Discharge", "Battery_SoC", "EffectiveProd", "ResidualDeficit"]].head(10)

,Date,ProdCombined,Baseload_MW,Battery_Charge,Battery_Discharge,Battery_SoC,EffectiveProd,ResidualDeficit
0,1990-01-01 00:00:00,3335.2,4574.18,0.00,0.00,0.00,3335.20,1238.98
1,1990-01-01 01:00:00,3284.6,4574.18,0.00,0.00,0.00,3284.60,1289.58
2,1990-01-01 02:00:00,3040.4,4574.18,0.00,0.00,0.00,3040.40,1533.78
3,1990-01-01 03:00:00,4468.2,4574.18,0.00,0.00,0.00,4468.20,105.98
4,1990-01-01 04:00:00,3757.6,4574.18,0.00,0.00,0.00,3757.60,816.58
5,1990-01-01 05:00:00,4637.6,4574.18,63.42,0.00,63.42,4574.18,0.00
6,1990-01-01 06:00:00,4809.2,4574.18,235.02,0.00,298.44,4574.18,0.00
7,1990-01-01 07:00:00,4303.2,4574.18,0.00,270.98,27.46,4574.18,0.00
8,1990-01-01 08:00:00,5161.2,4574.18,587.02,0.00,614.48,4574.18,0.00
9,1990-01-01 09:00:00,5407.6,4574.18,833.42,0.00,1447.90,4574.18,0.00


In [13]:
# 8a) Baseload coverage metrics before vs after storage

before_deficit = np.maximum(B - P, 0.0)
after_deficit = ResidualDeficit

hours_before = np.sum(before_deficit > 0)
hours_after = np.sum(after_deficit > 0)

energy_before = before_deficit.sum()
energy_after = after_deficit.sum()

print("Hours with deficit relative to baseload:")
print("- Before storage:", hours_before)
print("- After storage: ", hours_after)

print("\nTotal deficit energy (MWh-equivalent if P is MW):")
print("- Before storage:", round(energy_before, 2))
print("- After storage: ", round(energy_after, 2))

reduction_pct = 100 * (1 - energy_after / energy_before) if energy_before > 0 else 0
print(f"\nRelative reduction in deficit energy: {reduction_pct:.1f}%")

Hours with deficit relative to baseload:
- Before storage: 6365
- After storage:  4514

Total deficit energy (MWh-equivalent if P is MW):
- Before storage: 13152194.9
- After storage:  10420996.32

Relative reduction in deficit energy: 20.8%
